# Information Retrieval and Web Analytics

# Final Project - Part 3: Ranking & Filtering

In this notebook we:

1. Rank the results of the conjunctive (AND) search with **TF-IDF + cosine**, **BM25** and **our own score**,
   and compare the rankings.
2. Rank them with **word2vec + cosine** and return the **top-20** documents of our 5 queries of Part 2.

The code is in `myapp/search/ranking.py` (BM25, our score, word2vec and `search()`, which selects the method)
and `myapp/search/algorithms.py` (index, AND search and TF-IDF of Part 2).

### 1 - Packages

In [1]:
import os
import sys
import math
import time
import pickle

import numpy as np
import pandas as pd

# the notebook is in project_progress/part_3, so the root of the project is two folders up
ROOT = "../.." if not os.path.exists("myapp") else "."
sys.path.append(ROOT)

from myapp.preprocessing.text_processing import build_terms
from myapp.preprocessing.fields import load_raw_corpus, build_processed_corpus, save_processed_corpus, numeric_frame
from myapp.search.ranking import (METHODS, K1, B, bm25_term_weight, build_search_engine, search,
                                  percentile_in_results, rank_word2vec)
from myapp.evaluation.metrics import evaluate_query

CORPUS_PATH = os.path.join(ROOT, "data", "processed", "processed_corpus.pkl")
DATA_PATH = os.path.join(ROOT, "data", "fashion_products_dataset.json")
LABELS_PATH = os.path.join(ROOT, "data", "validation_labels.csv")
W2V_TOP20_PATH = os.path.join(ROOT, "project_progress", "part_3", "word2vec_top20.csv")
K = 10
pd.options.display.float_format = "{:.3f}".format
pd.options.display.max_colwidth = 60

### 2 - Load the processed corpus of Part 1

If `processed_corpus.pkl` does not exist yet, it is created with the Part 1 functions.

In [2]:
if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH, "rb") as fp:
        corpus = pickle.load(fp)
else:
    corpus = build_processed_corpus(load_raw_corpus(DATA_PATH))
    save_processed_corpus(corpus, CORPUS_PATH)

print("Number of documents: {}".format(len(corpus)))

Number of documents: 28080


### 3 - Build the search engine

`build_search_engine()` builds the inverted index of Part 2 (with df, idf and the tf-idf weights), the
statistics of BM25 (term frequencies, number of terms of each document and average length) and trains
word2vec on the same terms that are indexed. Then `search(query, engine, method)` runs the AND search and
ranks the results with `method` = `"tfidf"`, `"bm25"`, `"ours"` or `"word2vec"`.

In [3]:
start = time.time()
engine = build_search_engine(corpus)
print("Built in {:.1f} s".format(time.time() - start))
print("Terms in the index: {}".format(len(engine["index"])))
print("Average number of terms per document (L_ave): {:.2f}".format(engine["avg_len"]))
print("Terms with a word2vec vector (min_count = 2): {} of {}".format(len(engine["w2v"].wv), len(engine["index"])))

Built in 25.2 s
Terms in the index: 5733
Average number of terms per document (L_ave): 43.08
Terms with a word2vec vector (min_count = 2): 4453 of 5733


Our 5 test queries of Part 2 (ids 3-7) and the 2 validation queries (ids 1-2):

In [4]:
own_queries = {3: "women black round neck tshirt",
               4: "men checkered casual cotton shirt",
               5: "women printed kurta",
               6: "men white sneakers",
               7: "women v neck tshirt"}
validation_queries = {1: "women full sleeve sweatshirt cotton",
                      2: "men slim jeans blue"}

results = {q_id: {m: search(query, engine, m) for m in METHODS} for q_id, query in own_queries.items()}
for q_id, query in own_queries.items():
    print("Query {} '{}': {} documents".format(q_id, query, len(results[q_id]["tfidf"])))

Query 3 'women black round neck tshirt': 1112 documents
Query 4 'men checkered casual cotton shirt': 346 documents
Query 5 'women printed kurta': 42 documents
Query 6 'men white sneakers': 66 documents
Query 7 'women v neck tshirt': 6351 documents


## 1 - Three ways of ranking

### 4 - TF-IDF + cosine similarity

The same ranking as in Part 2 (`rank_documents` of `algorithms.py`):
$w_{t,d} = (1 + \log_2 f_{t,d}) \cdot \log_2(N/df_t)$ and
$score(q,d) = \sum_{t \in q} w_{t,q}\, w_{t,d} / (|q|\,|d|)$, where $|d|$ is the norm of all the tf-idf
weights of the document.

### 5 - BM25

Formula of the theory (Topic 6), with log base 2:

$$BM25(q,d) = \sum_{t \in q} \log_2\frac{N}{df_t} \cdot \frac{(k_1 + 1)\, tf_{t,d}}{k_1\left((1 - b) + b \frac{L_d}{L_{ave}}\right) + tf_{t,d}}$$

- $tf_{t,d}$: number of times the term appears in the document (number of positions in the index).
- $L_d$: number of terms of the document; $L_{ave}$: average over the corpus.
- $k_1 = 1.2$ (tf saturation) and $b = 0.75$ (length normalization): the typical values of the slides.
  We have no development collection to tune them.

We check the implementation with the example "president lincoln" of the slides.

In [5]:
# N = 500,000; df(president) = 40,000, df(lincoln) = 300; tf = 15 and 25; L_d / L_ave = 0.9; k1 = 1.2, b = 0.75
president = bm25_term_weight(15, math.log2(500000 / 40000), doc_len=0.9, avg_len=1)
lincoln = bm25_term_weight(25, math.log2(500000 / 300), doc_len=0.9, avg_len=1)
print("president: {:.2f} (slide 7.46)".format(president))
print("lincoln:   {:.2f} (slide 22.54)".format(lincoln))
print("BM25:      {:.2f} (slide 30.00, it rounds the intermediate values)".format(president + lincoln))

president: 7.46 (slide 7.46)
lincoln:   22.55 (slide 22.54)
BM25:      30.01 (slide 30.00, it rounds the intermediate values)


### 6 - Our score

$$ours(q,d) = BM25(q,d) \times title(q,d) \times stock(d) \times value(q,d)$$

- $title(q,d) = 0.5 + 0.5 \cdot \frac{\text{query terms in the title}}{\text{query terms}}$: the AND guarantees
  that all the terms are somewhere in the document, but not that the product **is** what the user asks for.
- $stock(d) = 1$ if the product is available, $0.5$ if it is out of stock (it is not removed: the AND search
  must return all the documents that contain the terms).
- $value(q,d) = 0.5 + 0.25 \cdot rating\_pct + 0.25 \cdot cheap\_pct$: percentiles of the rating (higher is
  better) and of the selling price (lower is better) **among the results of the same query**. A missing
  value gets 0.5. We do not use `discount`: it is computed from `actual_price`, which the seller sets.

Every factor is between 0.5 and 1, so the text score (BM25) can never be reduced to less than 1/8.

Some numbers of the corpus behind these choices:

In [6]:
numeric = numeric_frame(corpus)
print("Mean discount: {:.1f}% | products with a discount of 50% or more: {:.1f}%".format(
    numeric["discount"].mean(), 100 * (numeric["discount"] >= 50).mean()))
print("Out of stock: {:.1f}% | without rating: {:.1f}% | without selling price: {}".format(
    100 * numeric["out_of_stock"].mean(), 100 * numeric["average_rating"].isna().mean(),
    numeric["selling_price"].isna().sum()))

# percentiles: equal values share the percentile and missing values are neutral
print(percentile_in_results([3.0, 4.5, None, 4.5, 5.0], higher_is_better=True))
print(percentile_in_results([299, 999, 499], higher_is_better=False))

Mean discount: 50.3% | products with a discount of 50% or more: 58.8%
Out of stock: 5.9% | without rating: 8.1% | without selling price: 2
[0.  0.5 0.5 0.5 1. ]
[1.  0.  0.5]


### 7 - How the rankings differ

Top 10 of each query with the three scores:

In [7]:
def top_titles(q_id, n=10):
    table = {}
    for m in ["tfidf", "bm25", "ours"]:
        table[m] = [corpus[pid]["meta"]["title"][:45] for pid, score in results[q_id][m][:n]]
    return pd.DataFrame(table, index=range(1, n + 1))

for q_id, query in own_queries.items():
    print("\nQuery {}: {}".format(q_id, query))
    print(top_titles(q_id).to_string())


Query 3: women black round neck tshirt
                                     tfidf                                           bm25                                        ours
1     Solid Women Round Neck Black T-Shirt           Solid Women Round Neck Black T-Shirt      Printed Women Round Neck Black T-Shirt
2     Solid Women Round Neck Black T-Shirt  Striped Women Round Neck Black, White T-Shirt        Solid Women Round Neck Black T-Shirt
3     Solid Women Round Neck Black T-Shirt         Printed Women Round Neck Black T-Shirt        Solid Women Round Neck Black T-Shirt
4     Solid Women Round Neck Black T-Shirt         Printed Women Round Neck Black T-Shirt        Solid Women Round Neck Black T-Shirt
5     Solid Women Round Neck Black T-Shirt     Self Design Women Round Neck Black T-Shirt  Color Block Women Round Neck Black T-Shirt
6   Printed Women Round Neck Black T-Shirt     Color Block Women Round Neck Black T-Shirt        Solid Women Round Neck Black T-Shirt
7   Printed Women Roun

Summary for each query: documents in common in the top 10, mean number of terms of the top 10, different
scores in the top 20 (ties) and products out of stock in the top 10.

In [8]:
rows = []
for q_id, query in own_queries.items():
    res = results[q_id]
    top = {m: [pid for pid, score in res[m][:K]] for m in ["tfidf", "bm25", "ours"]}
    row = {"query": q_id,
           "common tfidf-bm25": len(set(top["tfidf"]) & set(top["bm25"])),
           "common bm25-ours": len(set(top["bm25"]) & set(top["ours"]))}
    for m in ["tfidf", "bm25", "ours"]:
        row["terms " + m] = np.mean([engine["doc_terms"][pid] for pid in top[m]])
    row["terms all results"] = np.mean([engine["doc_terms"][pid] for pid, score in res["tfidf"]])
    for m in ["tfidf", "bm25", "ours"]:
        row["scores " + m] = len(set(round(score, 9) for pid, score in res[m][:20]))
    for m in ["tfidf", "bm25", "ours"]:
        row["out of stock " + m] = sum(corpus[pid]["meta"]["out_of_stock"] for pid in top[m])
    rows.append(row)
comparison = pd.DataFrame(rows).set_index("query")
comparison.T

query,3,4,5,6,7
common tfidf-bm25,0.000,0.000,7.000,3.000,0.000
common bm25-ours,0.000,2.000,8.000,6.000,0.000
terms tfidf,23.800,18.300,31.200,51.100,23.600
terms bm25,30.700,25.400,32.600,38.700,44.100
terms ours,29.000,20.700,41.800,44.600,26.200
terms all results,51.634,31.997,62.190,73.394,51.409
scores tfidf,15.000,5.000,18.000,19.000,14.000
scores bm25,10.000,6.000,11.000,15.000,8.000
scores ours,18.000,9.000,10.000,17.000,11.000
out of stock tfidf,1.000,0.000,0.000,2.000,0.000


Why TF-IDF and BM25 order "men white sneakers" so differently: weight of each query term in the first
result of TF-IDF (a t-shirt) and in the first result of BM25 (a men's sneaker).

In [9]:
query_terms = build_terms(own_queries[6])
tshirt = results[6]["tfidf"][0][0]
sneaker = results[6]["bm25"][0][0]
rows = []
for pid in [tshirt, sneaker]:
    row = {"title": corpus[pid]["meta"]["title"], "terms": engine["doc_terms"][pid]}
    for t in query_terms:
        freq = engine["term_freq"][t].get(pid, 0)
        row["tf " + t] = freq
        row["tfidf w " + t] = engine["weights"][t].get(pid, 0)
        row["bm25 w " + t] = bm25_term_weight(freq, engine["idf"][t], engine["doc_terms"][pid], engine["avg_len"])
    for m in ["tfidf", "bm25"]:
        ranks = {p: i + 1 for i, (p, s) in enumerate(results[6][m])}
        row["rank " + m] = ranks[pid]
    rows.append(row)
pd.DataFrame(rows).T

,0,1
title,Solid Men Round Neck White T-Shirt,Sneakers For Men (White)
terms,53,37
tf men,2,3
tfidf w men,1.957,2.529
bm25 w men,1.263,1.585
tf white,4,2
tfidf w white,8.526,5.684
bm25 w white,4.625,4.069
tf sneaker,1,1
tfidf w sneaker,5.179,5.179


The same comparison on the judged documents of `validation_labels.csv` (as in Part 2: the 20 judged documents
are ranked, the ones that the AND search does not return are not in the ranking but count as relevant).
For our score, the percentiles are computed among all the results of the query, as in the real search.

In [10]:
labels_df = pd.read_csv(LABELS_PATH)
rows = []
for q_id, query in validation_queries.items():
    judged = labels_df[labels_df["query_id"] == q_id]
    labels = dict(zip(judged["pid"], judged["labels"]))
    for m in METHODS:
        returned = [labels[pid] for pid, score in search(query, engine, m) if pid in labels]
        metrics = evaluate_query(np.array(returned), K, sum(labels.values()))
        rows.append({"query": q_id, "method": m, **metrics})
validation_table = pd.DataFrame(rows).set_index(["query", "method"]).drop(columns="RR@10")
validation_table

P@10  R@10  AP@10  F1@10  NDCG@10
query method                                     
1     tfidf    1.000 0.769  0.769  0.870    1.000
      bm25     1.000 0.769  0.769  0.870    1.000
      ours     0.900 0.692  0.692  0.783    0.936
      word2vec 1.000 0.769  0.769  0.870    1.000
2     tfidf    0.600 0.600  0.569  0.600    0.716
      bm25     0.600 0.600  0.569  0.600    0.716
      ours     0.600 0.600  0.558  0.600    0.712
      word2vec 0.600 0.600  0.549  0.600    0.706

Which judged documents change position with our score in query 1:

In [11]:
judged = labels_df[labels_df["query_id"] == 1]
labels = dict(zip(judged["pid"], judged["labels"]))
rows = []
for m in ["bm25", "ours"]:
    ranking = [pid for pid, score in search(validation_queries[1], engine, m) if pid in labels]
    for rank, pid in enumerate(ranking[:K], start=1):
        meta = corpus[pid]["meta"]
        rows.append({"method": m, "rank": rank, "title": meta["title"][:50], "label": labels[pid],
                     "rating": meta["average_rating"], "price": meta["selling_price"]})
changes = pd.DataFrame(rows)
changes

,method,rank,title,label,rating,price
0,bm25,1,Full Sleeve Printed Women Sweatshirt,1,3.200,999.000
1,bm25,2,Full Sleeve Color Block Women Sweatshirt,1,3.600,349.000
2,bm25,3,Full Sleeve Graphic Print Women Sweatshirt,1,2.700,699.000
3,bm25,4,Full Sleeve Solid Women Sweatshirt,1,4.300,759.000
4,bm25,5,Full Sleeve Solid Women Sweatshirt,1,4.000,598.000
5,bm25,6,Full Sleeve Graphic Print Women Sweatshirt,1,3.900,849.000
6,bm25,7,Full Sleeve Graphic Print Women Sweatshirt,1,3.300,849.000
7,bm25,8,Full Sleeve Graphic Print Women Sweatshirt,1,NaN,799.000
8,bm25,9,Full Sleeve Graphic Print Women Sweatshirt,1,3.900,899.000
9,bm25,10,Full Sleeve Printed Women Sweatshirt,1,4.300,2099.000


## 2 - word2vec + cosine similarity

### 8 - Train word2vec

We train word2vec (CBOW of gensim, 100 dimensions, window 5, 20 epochs) on the `search_tokens` of the documents,
the same terms that are indexed. Terms that appear only once in the corpus are ignored (`min_count = 2`): one
occurrence is not enough to learn a vector, so these terms have no vector and are skipped when a text vector is
computed. With one worker, a fixed seed and a stable hash the vectors are the same in every run. A text is
represented by the **average** of the vectors of its terms, and the AND results are ranked by the cosine between the
query vector and the document vector.

All the terms of our 7 queries (ids 1-7) have a vector:

In [12]:
for q_id, query in {**validation_queries, **own_queries}.items():
    terms = build_terms(query)
    missing = [t for t in terms if t not in engine["w2v"].wv]
    print("Query {} {}: missing vectors = {}".format(q_id, terms, missing))

Query 1 ['women', 'full', 'sleev', 'sweatshirt', 'cotton']: missing vectors = []
Query 2 ['men', 'slim', 'jean', 'blue']: missing vectors = []
Query 3 ['women', 'black', 'round', 'neck', 'tshirt']: missing vectors = []
Query 4 ['men', 'checker', 'casual', 'cotton', 'shirt']: missing vectors = []
Query 5 ['women', 'print', 'kurta']: missing vectors = []
Query 6 ['men', 'white', 'sneaker']: missing vectors = []
Query 7 ['women', 'neck', 'tshirt']: missing vectors = []


Most similar words of some query terms:

In [13]:
w2v = engine["w2v"]
for word in ["sneaker", "kurta", "white", "black", "checker"]:
    print(word, "->", ", ".join("{} ({:.2f})".format(w, s) for w, s in w2v.wv.most_similar(word, topn=5)))

sneaker -> complet (0.48), throw (0.47), segment (0.47), score (0.46), pair (0.46)
kurta -> chines (0.39), jute (0.38), shirt (0.36), blazer (0.35), rope (0.33)
white -> red (0.90), black (0.88), grey (0.85), yellow (0.84), maroon (0.81)
black -> grey (0.89), white (0.88), red (0.86), maroon (0.80), yellow (0.80)
checker -> solid (0.69), stripe (0.66), print (0.60), spread (0.49), floral (0.48)


### 9 - Top-20 documents of our 5 queries with word2vec + cosine

In [14]:
rows = []
for q_id, query in own_queries.items():
    for rank, (pid, score) in enumerate(results[q_id]["word2vec"][:20], start=1):
        rows.append({"query_id": q_id, "query": query, "rank": rank, "pid": pid,
                     "title": corpus[pid]["meta"]["title"], "score": round(score, 4)})
w2v_top20 = pd.DataFrame(rows)
w2v_top20.to_csv(W2V_TOP20_PATH, index=False)
print("Saved in", W2V_TOP20_PATH)

for q_id, query in own_queries.items():
    print("\nQuery {}: {}".format(q_id, query))
    print(w2v_top20[w2v_top20["query_id"] == q_id][["rank", "pid", "title", "score"]].to_string(index=False))

Saved in ../../project_progress/part_3/word2vec_top20.csv

Query 3: women black round neck tshirt
 rank              pid                                         title  score
    1 TSHFYFZRJHBV79DP        Printed Women Round Neck Black T-Shirt  0.977
    2 TSHFYYRA3SYFMC9T        Printed Women Round Neck Black T-Shirt  0.977
    3 TSHFYYRC49HQDX96        Printed Women Round Neck Black T-Shirt  0.977
    4 TSHFYYXDDGGGGKKZ        Printed Women Round Neck Black T-Shirt  0.977
    5 TSHFNT4FBEJJZ8HB      Superhero Women Round Neck Black T-Shirt  0.976
    6 TSHFVTWHMCHZTHNC     Typography Women Round Neck Black T-Shirt  0.976
    7 TSHFYUQYXW4BY9GR Striped Women Round Neck Black, White T-Shirt  0.975
    8 TSHFW4S4JVEZWDAM      Superhero Women Round Neck Black T-Shirt  0.975
    9 TSHFZWJMK4HG7R4X        Printed Women Round Neck Black T-Shirt  0.975
   10 TSHFVSS3GR3KV6YT        Printed Women Round Neck Black T-Shirt  0.975
   11 TSHFVTXFH3JSZPUG     Typography Women Round Neck Black T-Shi

Comparison with TF-IDF: documents in common in the top 20 and mean number of terms of the top 20.

In [15]:
rows = []
for q_id in own_queries:
    top_w2v = [pid for pid, score in results[q_id]["word2vec"][:20]]
    top_tfidf = [pid for pid, score in results[q_id]["tfidf"][:20]]
    rows.append({"query": q_id,
                 "common with tfidf": len(set(top_w2v) & set(top_tfidf)),
                 "terms word2vec": np.mean([engine["doc_terms"][pid] for pid in top_w2v]),
                 "terms tfidf": np.mean([engine["doc_terms"][pid] for pid in top_tfidf]),
                 "sneakers in top 20": sum("Sneakers" in corpus[pid]["meta"]["title"] for pid in top_w2v) if q_id == 6 else None,
                 "V neck in top 20": sum("v neck" in corpus[pid]["meta"]["title"].lower().replace("-", " ") for pid in top_w2v) if q_id == 7 else None})
pd.DataFrame(rows).set_index("query")

,common with tfidf,terms word2vec,terms tfidf,sneakers in top 20,V neck in top 20
query,,,,,
3,1,24.550,24.200,NaN,NaN
4,0,27.150,18.650,NaN,NaN
5,13,55.250,38.750,NaN,NaN
6,17,58.250,53.300,0.000,NaN
7,0,41.350,23.950,NaN,0.000


The query vector is the average of only 3-5 terms, and every result contains all of them (AND), so the order
depends on the other words of each document. For example, in query 6 the cosine of the query vector with the
vectors of its own terms:

In [16]:
query_vector = np.mean([w2v.wv[t] for t in build_terms(own_queries[6])], axis=0)
for t in build_terms(own_queries[6]) + ["tshirt", "shoe"]:
    v = w2v.wv[t]
    print("{:8s} {:.3f}".format(t, float(np.dot(query_vector, v) / (np.linalg.norm(query_vector) * np.linalg.norm(v)))))

men      0.692
white    0.604
sneaker  0.760
tshirt   0.397
shoe     0.016
